# RAG Pipeline — Embeddings, Vector Search & LLM

### What we'll build:
```
Document → Chunking → Embeddings → Vector DB
                                        ↑
User Query → Embedding → Semantic Search (Top-K / Top-P) → LLM → Answer
```

In [13]:
# Install required libraries
# Run this once, then restart kernel

!pip install openai faiss-cpu tiktoken python-dotenv langchain langchain-openai langchain-community

In [14]:
from dotenv import load_dotenv

load_dotenv()

True

---
## Step 1 — The Document

This is the knowledge source we want to query.
In a real app this could be a PDF, website, or database.

In [15]:
# Our sample document — a short article about the Solar System
# In a real project you'd load this from a file:
# with open("my_document.txt") as f: document = f.read()

document = """
The Solar System consists of the Sun and everything bound to it by gravity.
This includes eight planets, their moons, and countless smaller objects like asteroids and comets.

Mercury is the closest planet to the Sun. It has no atmosphere and extreme temperature swings,
from -180°C at night to 430°C during the day. Mercury completes an orbit in just 88 Earth days.

Venus is the second planet and the hottest in the Solar System due to its thick CO2 atmosphere
that creates a runaway greenhouse effect. Surface temperatures reach 465°C.

Earth is the third planet and the only known planet to support life. It has liquid water,
a breathable atmosphere, and a magnetic field that protects it from solar radiation.
Earth has one natural satellite — the Moon.

Mars is the fourth planet, known as the Red Planet due to iron oxide on its surface.
It has the tallest volcano in the Solar System — Olympus Mons — standing 21 km high.
Mars has two small moons: Phobos and Deimos.

Jupiter is the largest planet, a gas giant with a mass greater than all other planets combined.
Its Great Red Spot is a storm that has been raging for hundreds of years.
Jupiter has at least 95 known moons, including Ganymede — the largest moon in the Solar System.

Saturn is famous for its stunning ring system made of ice and rock particles.
It is the least dense planet — it could float on water. Saturn has 146 known moons,
including Titan, which has a thick atmosphere and lakes of liquid methane.

Uranus is an ice giant that rotates on its side, with an axial tilt of 98 degrees.
It appears blue-green due to methane in its atmosphere.

Neptune is the farthest planet, with the strongest winds in the Solar System reaching 2,100 km/h.
Its largest moon, Triton, orbits in the opposite direction to Neptune's rotation.

Beyond Neptune lies the Kuiper Belt, home to dwarf planets like Pluto, Eris, and Makemake.
Even further is the Oort Cloud, a vast shell of icy objects that is the source of long-period comets.
"""

print(f"Document length: {len(document)} characters")

Document length: 1996 characters


---
## Step 2 — Chunking

**Why chunk?**  
LLMs and embedding models have token limits. Also, smaller focused chunks 
give more precise search results than embedding a huge document all at once.

**Key parameters:**
- `chunk_size` — max characters per chunk
- `chunk_overlap` — how many characters repeat between chunks (preserves context at boundaries)

In [16]:
!pip install langchain-text-splitters

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries to split on paragraphs → sentences → words
# in that order, keeping chunks as semantically whole as possible

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # each chunk is max 300 characters
    chunk_overlap=50,     # 50 characters repeat between consecutive chunks
                          # this prevents losing context at chunk boundaries
    separators=["\n\n", "\n", ". ", " "]  # tries these splits in order
)

chunks = splitter.split_text(document)

print(f"Total chunks created: {len(chunks)}")
print("\n" + "="*60)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)

Total chunks created: 10


--- Chunk 1 (174 chars) ---
The Solar System consists of the Sun and everything bound to it by gravity.
This includes eight planets, their moons, and countless smaller objects like asteroids and comets.

--- Chunk 2 (190 chars) ---
Mercury is the closest planet to the Sun. It has no atmosphere and extreme temperature swings,
from -180°C at night to 430°C during the day. Mercury completes an orbit in just 88 Earth days.

--- Chunk 3 (170 chars) ---
Venus is the second planet and the hottest in the Solar System due to its thick CO2 atmosphere
that creates a runaway greenhouse effect. Surface temperatures reach 465°C.

--- Chunk 4 (218 chars) ---
Earth is the third planet and the only known planet to support life. It has liquid water,
a breathable atmosphere, and a magnetic field that protects it from solar radiation.
Earth has one natural satellite — the Moon.

--- Chunk 5 (214 chars) ---
Mars is the fourth planet, known as the Red Planet due to iron oxide on i

---
## Step 3 — Creating Embeddings

**What is an embedding?**  
An embedding converts text into a list of numbers (a vector) that captures its *meaning*.  
Similar sentences produce vectors that are close together in space.

```
"hottest planet"  → [0.12, -0.45, 0.89, ...]  ← close to ↓
"Venus temperature" → [0.14, -0.41, 0.91, ...]

"fastest car"  → [-0.72, 0.33, -0.11, ...]  ← far from both above
```

We use OpenAI's `text-embedding-3-small` — fast, cheap, and 1536 dimensions.

In [18]:
from langchain_openai import OpenAIEmbeddings
import httpx

# Initialize the embedding model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 1536-dimensional vectors
                                    # text-embedding-3-large is more powerful but slower/pricier
)

# Let's see what a single embedding looks like

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    http_client=httpx.Client(verify=False)  # 👈 disables SSL check
)

sample_embedding = embedding_model.embed_query(chunks[0])
print(f"Embedding dimensions: {len(sample_embedding)}")
print(f"First 10 values: {sample_embedding[:10]}")
print("\n(Each chunk will become a vector like this and be stored in the database)")

Embedding dimensions: 1536
First 10 values: [-0.0275421142578125, 0.0174102783203125, 0.03717041015625, 0.042877197265625, -0.0013027191162109375, 0.043243408203125, 0.0126495361328125, -0.0139007568359375, 0.0175628662109375, 0.00010019540786743164]

(Each chunk will become a vector like this and be stored in the database)


---
## Step 4 — Storing in a Vector Database (FAISS)

**FAISS** (Facebook AI Similarity Search) is an in-memory vector database.  
It stores all chunk embeddings and lets us search them ultra-fast.

Under the hood:
1. Each chunk is converted to an embedding vector
2. All vectors are indexed in FAISS
3. On search, query vector is compared to all stored vectors using cosine similarity

In [19]:
from langchain_community.vectorstores import FAISS

# This single line:
# 1. Calls OpenAI API to embed every chunk
# 2. Stores all vectors + original text in FAISS

print("Embedding all chunks and storing in FAISS...")

vector_db = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

# vector_db.save_local("faiss_index")  # creates a folder called 'faiss_index'


print(f"✅ Vector DB ready! {len(chunks)} chunks stored.")
print(f"Each stored as a {len(sample_embedding)}-dimensional vector.")

Embedding all chunks and storing in FAISS...
✅ Vector DB ready! 10 chunks stored.
Each stored as a 1536-dimensional vector.


---
## Step 5 — Top-K Retrieval (with Scores)

**Top-K** = retrieve the K most similar chunks to the query.

How it works:
1. Convert user query → embedding vector
2. Compute similarity between query vector and all stored vectors
3. Return the K closest ones

**Score** = L2 distance (lower = more similar / better match)

In [20]:
user_query = "Which planet is the hottest and why?"

K = 5  # retrieve top 5 most relevant chunks

# similarity_search_with_score returns (Document, score) tuples
# Score is L2 distance — LOWER score = MORE similar
top_k_results = vector_db.similarity_search_with_score(
    query=user_query,
    k=K
)

print(f"Query: '{user_query}'")
print(f"\n{'='*60}")
print(f"TOP-{K} RESULTS (sorted by similarity, best first)")
print(f"{'='*60}")

for rank, (doc, score) in enumerate(top_k_results, 1):
    print(f"\nRank #{rank} | Score: {score:.4f} (lower = more relevant)")
    print(f"Chunk: {doc.page_content}")
    print("-"*60)

Query: 'Which planet is the hottest and why?'

TOP-5 RESULTS (sorted by similarity, best first)

Rank #1 | Score: 0.6744 (lower = more relevant)
Chunk: Venus is the second planet and the hottest in the Solar System due to its thick CO2 atmosphere
that creates a runaway greenhouse effect. Surface temperatures reach 465°C.
------------------------------------------------------------

Rank #2 | Score: 0.9532 (lower = more relevant)
Chunk: Mercury is the closest planet to the Sun. It has no atmosphere and extreme temperature swings,
from -180°C at night to 430°C during the day. Mercury completes an orbit in just 88 Earth days.
------------------------------------------------------------

Rank #3 | Score: 1.0421 (lower = more relevant)
Chunk: Jupiter is the largest planet, a gas giant with a mass greater than all other planets combined.
Its Great Red Spot is a storm that has been raging for hundreds of years.
Jupiter has at least 95 known moons, including Ganymede — the largest moon in the 

---
## Step 6 — Top-P Retrieval

**Top-P** = instead of a fixed count (K), keep chunks until their *cumulative score proportion* reaches P.

Think of it like this:
- Convert raw scores to proportional weights (lower score → higher weight)
- Keep adding chunks until we've covered P% of the total "relevance mass"
- P=0.5 means: take enough top chunks to cover 50% of total relevance

**Why use Top-P?**  
Top-K always returns exactly K chunks even if some are irrelevant.  
Top-P adapts — returns fewer chunks when one is a strong match, more when relevance is spread out.

In [21]:
import numpy as np

def top_p_retrieval(vector_db, query, p=0.5, max_results=10):
    """
    Retrieve chunks using Top-P (nucleus) sampling.
    
    Args:
        vector_db   : FAISS vector store
        query       : user's question string
        p           : cumulative relevance threshold (0.0 to 1.0)
        max_results : fetch this many candidates first, then filter by p
    
    Returns:
        list of (doc, score, weight) tuples
    """
    # Step 1: fetch a pool of candidates
    candidates = vector_db.similarity_search_with_score(query, k=max_results)
    
    # Step 2: extract scores (L2 distances — lower is better)
    scores = np.array([score for _, score in candidates])
    
    # Step 3: convert distances to weights
    # We invert: weight = 1 / (1 + distance) so closer = higher weight
    weights = 1 / (1 + scores)
    
    # Step 4: normalize weights to sum to 1 (like probabilities)
    weights_normalized = weights / weights.sum()
    
    # Step 5: keep adding top results until cumulative weight >= p
    cumulative = 0.0
    selected = []
    
    for i, (doc, score) in enumerate(candidates):
        cumulative += weights_normalized[i]
        selected.append((doc, score, weights_normalized[i]))
        if cumulative >= p:
            break
    
    return selected, weights_normalized


# Run Top-P retrieval
P = 0.5  # cover 50% of relevance mass

top_p_results, all_weights = top_p_retrieval(vector_db, user_query, p=P)

print(f"Query: '{user_query}'")
print(f"P = {P} (stop when cumulative relevance reaches {int(P*100)}%)")
print(f"\n{'='*60}")
print(f"TOP-P RESULTS → selected {len(top_p_results)} out of {len(all_weights)} candidates")
print(f"{'='*60}")

cumulative_display = 0
for rank, (doc, score, weight) in enumerate(top_p_results, 1):
    cumulative_display += weight
    print(f"\nRank #{rank} | Score: {score:.4f} | Weight: {weight:.4f} | Cumulative: {cumulative_display:.4f}")
    print(f"Chunk: {doc.page_content}")
    print("-"*60)

print(f"\n✅ Stopped at cumulative weight {cumulative_display:.4f} (>= p={P})")

Query: 'Which planet is the hottest and why?'
P = 0.5 (stop when cumulative relevance reaches 50%)

TOP-P RESULTS → selected 5 out of 10 candidates

Rank #1 | Score: 0.6744 | Weight: 0.1238 | Cumulative: 0.1238
Chunk: Venus is the second planet and the hottest in the Solar System due to its thick CO2 atmosphere
that creates a runaway greenhouse effect. Surface temperatures reach 465°C.
------------------------------------------------------------

Rank #2 | Score: 0.9532 | Weight: 0.1062 | Cumulative: 0.2300
Chunk: Mercury is the closest planet to the Sun. It has no atmosphere and extreme temperature swings,
from -180°C at night to 430°C during the day. Mercury completes an orbit in just 88 Earth days.
------------------------------------------------------------

Rank #3 | Score: 1.0421 | Weight: 0.1015 | Cumulative: 0.3316
Chunk: Jupiter is the largest planet, a gas giant with a mass greater than all other planets combined.
Its Great Red Spot is a storm that has been raging for hundred

---
## Step 7 — Compare Top-K vs Top-P

In [22]:
print("COMPARISON: Top-K vs Top-P")
print("="*60)
print(f"Top-K (k=5)  → always returns exactly 5 chunks")
print(f"Top-P (p=0.5)→ returned {len(top_p_results)} chunk(s) to cover 50% of relevance")
print()
print("When to use which:")
print("  Top-K  → when you always want a fixed number of results")
print("  Top-P  → when you want to adapt based on how confident the matches are")
print("           (one great match = fewer chunks; many similar matches = more chunks)")

COMPARISON: Top-K vs Top-P
Top-K (k=5)  → always returns exactly 5 chunks
Top-P (p=0.5)→ returned 5 chunk(s) to cover 50% of relevance

When to use which:
  Top-K  → when you always want a fixed number of results
  Top-P  → when you want to adapt based on how confident the matches are
           (one great match = fewer chunks; many similar matches = more chunks)


---
## Step 8 — Send to LLM and Get Final Answer

Now we take the Top-K retrieved chunks, inject them as **context** into the LLM prompt,
and ask it to answer the question.

This is the **RAG** (Retrieval-Augmented Generation) pattern:
```
Answer = LLM(question + retrieved_context)
```

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def ask_with_rag(query, vector_db, k=5):
    """
    Full RAG pipeline:
    1. Embed the query
    2. Retrieve top-K relevant chunks
    3. Build a prompt with context
    4. Get LLM answer
    """
    
    # --- Step A: Retrieve relevant chunks ---
    print(f"🔍 Searching for: '{query}'")
    results = vector_db.similarity_search_with_score(query, k=k)
    
    # Build context string from retrieved chunks
    context = "\n\n".join([
        f"[Chunk {i+1} | Score: {score:.4f}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    ])
    
    print(f"📚 Retrieved {len(results)} chunks")
    print("-"*60)
    
    # --- Step B: Build the prompt ---
    system_prompt = """You are a helpful assistant. Answer the user's question 
using ONLY the context provided below. If the answer is not in the context, 
say 'I don't have enough information to answer that.'
Be concise and accurate."""
    
    user_prompt = f"""Context:
{context}

Question: {query}"""
    
    # --- Step C: Send to LLM ---
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]
    
    response = llm.invoke(messages)
    
    return response.content


# Ask our question!
answer = ask_with_rag(user_query, vector_db, k=5)

print(f"\n🤖 LLM ANSWER:")
print("="*60)
print(answer)

🔍 Searching for: 'Which planet is the hottest and why?'
📚 Retrieved 5 chunks
------------------------------------------------------------

🤖 LLM ANSWER:
Venus is the hottest planet in the Solar System due to its thick CO2 atmosphere that creates a runaway greenhouse effect, resulting in surface temperatures reaching 465°C.


---
## Step 9 — Try Different Queries

In [ ]:
queries = [
    "How many moons does Jupiter have?",
    "What makes Saturn special compared to other planets?",
    "Which planet has the strongest winds?",
    "What is beyond Neptune?"
]

for q in queries:
    print(f"\n❓ {q}")
    answer = ask_with_rag(q, vector_db, k=3)
    print(f"💬 {answer}")
    print("="*60)


❓ How many moons does Jupiter have?
🔍 Searching for: 'How many moons does Jupiter have?'
📚 Retrieved 3 chunks
------------------------------------------------------------
💬 Jupiter has at least 95 known moons.

❓ What makes Saturn special compared to other planets?
🔍 Searching for: 'What makes Saturn special compared to other planets?'
📚 Retrieved 3 chunks
------------------------------------------------------------
💬 Saturn is special because of its stunning ring system made of ice and rock particles, it is the least dense planet and could float on water, and it has 146 known moons, including Titan, which has a thick atmosphere and lakes of liquid methane.

❓ Which planet has the strongest winds?
🔍 Searching for: 'Which planet has the strongest winds?'
📚 Retrieved 3 chunks
------------------------------------------------------------
💬 Neptune has the strongest winds in the Solar System, reaching 2,100 km/h.

❓ What is beyond Neptune?
🔍 Searching for: 'What is beyond Neptune?'
📚 Retri

: 

---
## Summary — Full Pipeline

```
INDEXING (done once):
Document → RecursiveCharacterTextSplitter → chunks
chunks   → OpenAI text-embedding-3-small  → vectors
vectors  → FAISS.from_texts               → vector DB

QUERYING (done per question):
User query → embed_query → query vector
query vector → similarity_search → Top-K chunks (with scores)
             OR top_p_retrieval  → Top-P chunks (adaptive)
chunks + query → LLM prompt → Final Answer
```

| Concept | What it does |
|---|---|
| **Chunking** | Splits large doc into searchable pieces |
| **Embeddings** | Converts text to vectors that capture meaning |
| **FAISS** | Stores & searches vectors efficiently |
| **Top-K** | Fixed number of best results |
| **Top-P** | Adaptive results based on relevance mass |
| **RAG** | LLM answers using retrieved context, not just its training data |